# SES_Achievement_Gap_Terrain_Example

Notebook companion to `generate_ses_gap_terrain.py` -- "the forest, not the trees": two stacked GlyphViz `TOPO_SURFACE` meshes over the same (grade, year) grid, from the same San Diego Unified STAR testing database as `CalStateTesting_Example`.

One mesh is economically-disadvantaged students' average CST proficiency (`subgroup` code 31), the other is non-disadvantaged (code 111). Both sit on the same XY grid (grade 2-11 x year 2003-2012) -- the **vertical gap between the two surfaces at any point IS the achievement gap**, legible without reading an axis label, while each surface's own slope still shows that group's real trend over the decade.

Real numbers behind this: a persistent 15-35 percentage-point gap, every grade, every year -- both groups improved 2003-2012, but the gap barely closed.

This generalizes to any other binary subgroup split in the same data (disability status, English-learner status, ...) via `--group-a-code`/`--group-b-code` -- see the parameters cell below.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from generate_ses_gap_terrain import build_arg_parser, generate

## Parameters

Edit this list and re-run -- same flags as the CLI (`python generate_ses_gap_terrain.py --help` for the full list). `--source mysql` requires a reachable MySQL server with the `calstatetesting_all` database; `--source csv` reads `sdusd<year>.csv` files from `--data-dir` instead.

In [ ]:
cli_args = [
    "--source", "mysql",
    "--start-year", "2003",
    "--end-year", "2012",
    "--prefix", "SES_Achievement_Gap_Terrain_Example_notebook",
]

args = build_arg_parser().parse_args(cli_args)
args

## Generate the node/tag CSVs

In [ ]:
node_path, tag_path = generate(args)
node_path, tag_path

## Peek at the output: the gap itself, grade x year

In [ ]:
import pandas as pd

tags_df = pd.read_csv(tag_path)

# Tag titles look like "Grade 8 | 2010 | Economically Disadvantaged: 39.1% above proficient"
parsed = tags_df["title"].str.extract(
    r"Grade (?P<grade>\d+) \| (?P<year>\d+) \| (?P<label>[^:]+): (?P<pct>[\d.]+)% above proficient"
)
parsed["grade"] = parsed["grade"].astype(int)
parsed["year"] = parsed["year"].astype(int)
parsed["pct"] = parsed["pct"].astype(float)

pivot = parsed.pivot_table(index="year", columns=["grade", "label"], values="pct")
labels = parsed["label"].unique()
gap = pivot.xs(labels[1], level="label", axis=1) - pivot.xs(labels[0], level="label", axis=1)
print(f"Achievement gap ({labels[1]} minus {labels[0]}), percentage points, by year x grade:")
gap.round(1)

## Optional: render a quick preview

Same headless-GL pattern as `tools/export_frame.py`. Camera angle matters a lot for legibility here -- a 3/4 elevated view reads as two distinct terrains; a near-horizontal side view compresses the rows and reads more like skyscrapers. Tags are dense at this many nodes (200 labels) -- set `vp.show_tags = False` for a clean shot.

In [ ]:
REPO_ROOT = HERE.parent.parent  # examples/SES_Achievement_Gap_Terrain_Example -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from PySide6.QtWidgets import QApplication

app = QApplication.instance() or QApplication(sys.argv[:1])

from glyphviz_core.csv_reader import load_node_csv
from glyphviz_gl.viewport import Viewport

nodes = load_node_csv(str(node_path))
vp = Viewport()
vp.set_nodes(nodes)
vp.show_tags = False
vp.set_camera(azimuth=35, elevation=30, distance=130)

preview_path = HERE / "notebook_preview.png"
vp.export_png(str(preview_path), width=960, height=540)

from IPython.display import Image
Image(filename=str(preview_path))

## Load in GlyphViz

File > Open Node CSV -> the `node_path` printed above. Orbit to a 3/4 elevated view for the clearest read of the two terrains.